In [1]:
import pandas as pd 
from datetime import date, timedelta
import warnings
from finvizfinance.screener.overview import Overview
import plotly.express as px

# solution if plotly does not plot in Jupyter
# conda install -c conda-forge nodejs
# conda install -c conda-forge/label/gcc7 nodejs
# conda install -c conda-forge/label/cf201901 nodejs
# conda install -c conda-forge/label/cf202003 nodejs
# jupyter labextension install jupyterlab-plotly
# !jupyter labextension install jupyterlab-plotly

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.3f' % x)


In [2]:
%cd /Users/safishajjouz/GitHub/QuantitativePortfolioManagement

/Users/safishajjouz/GitHub/QuantitativePortfolioManagement


In [4]:
from myPortfolioManagement.myData import get_sp500_tickers, get_stock_prices
from myPortfolioManagement.myPerformanceMetrics import performance_overview

In [5]:
# for filtering: https://finviz.com/screener.ashx
df = get_sp500_tickers()
df = df[df['Industry']!='Exchange Traded Fund']

In [6]:
df_larg_caps = df [ df['Market Cap']>df['Market Cap'].mean()]

In [7]:
df_larg_caps.sort_values('P/E', ascending = False)

,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume
28,AMD,Advanced Micro Devices Inc.,Technology,Semiconductors,USA,191550000000.000,935.090,118.570,-0.023,39854416.000
356,PANW,Palo Alto Networks Inc,Technology,Software - Infrastructure,USA,90370000000.000,162.470,286.610,-0.032,4642133.000
32,AMT,American Tower Corp.,Real Estate,REIT - Specialty,USA,97300000000.000,136.810,208.730,-0.008,2668581.000
280,LLY,Lilly(Eli) & Co,Healthcare,Drug Manufacturers - General,USA,555990000000.000,108.110,585.680,0.003,2577122.000
116,CRM,Salesforce Inc,Technology,Software - Application,USA,243890000000.000,95.150,250.660,-0.036,11429249.000
...,...,...,...,...,...,...,...,...,...,...
475,VZ,Verizon Communications Inc,Communication Services,Telecom Services,USA,159920000000.000,7.670,38.040,-0.014,22109240.000
75,C,Citigroup Inc,Financial,Banks - Diversified,USA,90660000000.000,7.520,47.370,0.003,19351024.000
51,BA,Boeing Co.,Industrials,Aerospace & Defense,USA,142090000000.000,NaN,234.870,0.004,5228476.000
241,INTC,Intel Corp.,Technology,Semiconductors,USA,178550000000.000,NaN,42.350,-0.032,50417588.000


In [ ]:
# Set dates

start_date = '1995-01-01'
end_date = date.today() - timedelta(days=1)
end_date = end_date.strftime("%Y-%m-%d")

df_price = get_stock_prices(yahoo_tickers=df_larg_caps['Ticker'].to_list(),
                            start_date=start_date,
                            end_date=end_date, 
                            wide_format = True,
                            time_interval='daily')




In [46]:
 # get performance and merge with other info 
df_perf_overview = performance_overview(df_price, prices=True, short = False)
df_performance = df_perf_overview[['cagr', 'ytd']]
df_performance.index = df_performance.index.set_names('Ticker')
df_performance = df_performance.reset_index().merge(df_larg_caps)
df_performance['Market Cap'] = df_performance['Market Cap']/1000000000

In [47]:
df_performance

,Ticker,cagr,ytd,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume
0,AAPL,0.244,0.037,Apple Inc.,Technology,Consumer Electronics,USA,2067.700,22.080,134.760,0.010,57712715.000
1,ABBV,0.209,-0.041,AbbVie Inc.,Healthcare,Drug Manufacturers - General,USA,266.650,20.460,153.600,0.009,6017006.000
2,ABT,0.128,0.039,Abbott Laboratories,Healthcare,Medical Devices,USA,195.220,25.500,113.510,0.019,5345534.000
3,ACN,0.163,0.062,Accenture plc,Technology,Information Technology Services,Ireland,181.460,25.620,282.140,-0.006,2226663.000
4,ADBE,0.178,0.023,Adobe Inc.,Technology,Software - Infrastructure,USA,158.030,34.090,344.380,-0.001,2581506.000
...,...,...,...,...,...,...,...,...,...,...,...,...
102,VZ,0.069,0.079,Verizon Communications Inc.,Communication Services,Telecom Services,USA,173.760,9.100,41.860,0.001,17224498.000
103,WFC,0.105,0.071,Wells Fargo & Company,Financial,Banks - Diversified,USA,167.000,11.490,44.220,0.033,41522196.000
104,WMT,0.115,0.025,Walmart Inc.,Consumer Defensive,Discount Stores,USA,382.450,44.900,145.290,0.003,4566406.000
105,XOM,0.108,0.026,Exxon Mobil Corporation,Energy,Oil & Gas Integrated,USA,460.390,9.230,113.150,-0.001,12005237.000


In [48]:
fig = px.scatter(df_performance, x="ytd", y="cagr", width=1200,
                 height=1100,
                 size ='Market Cap', template='plotly_dark', text='Company',
                 color_continuous_scale=px.colors.sequential.Viridis, trendline="ols",
                 title = 'S&P 500 Large Caps: CAGR vs YTD')
fig.show()